```
<01_object_detection.ipynb>

제미나이 의존도: 80-90%

객체탐지. 저번에 한번 찍먹을 해봤었지만, 여전히 낯설게 느껴졌다. 크롤링해서 구한 이미지로 roboflow 사이트에서 네모박스도 쳐보았다.
초반에 힘을 빼면 지치기 때문에 15장으로만 해봤다. 주제는 "길거리에 버려진 쓰레기들"
많은 쓰레기 종류가 있지만 일단 캔류, 페트병, 쓰레기 이렇게만 분류했다.

객체 탐지는 이미지 분류와는 다르게 네모박스의 정보를 담는 .txt 파일이 이미지마다 하나씩 존재한다.
클래스번호, 중앙x값, 중앙y값, 가로, 세로 값이 0~1의 값으로 정규화되어 세팅되어 있다.
이것을 가공해서 (x_min, y_min), (x_max, y_max) 픽셀값을 구한다.

데이터셋 구조도 이미지 분류와는 조금 다르다.
cv2로 변환한 이미지 객체, target이 있는데 target에는 이미지에 있는 네모박스의 좌표 정보, 클래스번호 값이 있다.

이미지마다 네모박스의 개수가 다르기 때문에 이미지 분류 때 했던 것처럼 배치 사이즈로 묶을 수가 없다.
그래서 collate_fn이라는 함수를 사용하는데, 파이썬 리스트 구조로 묶어서 전달해주는 역할을 한다.
그래서 실제로 dataloader에서 뽑아 학습할 때도 리스트 컴프리헨션을 통해 개별 텐서 요소들을 하나씩 꺼내서 사용해야 한다.

객체 탐지.. 계속 하면서 제미나이 의존도를 많이 낮춰봅시다. 렛츠고.
```

In [15]:
import cv2
import os

# 반복문을 사용해서 모든 이미지에 대한 네모박스 이미지 만들어내기.

common_path = "object_detection/1st_prj/train"
image_path = os.path.join(common_path, "images")
label_path = os.path.join(common_path, "labels")

for image_name in os.listdir(image_path):
    if image_name.endswith(('jpg', 'jpeg', 'png')):
        base_name, ext = os.path.splitext(image_name)
        # print(base_name) # 123
        full_image_path = os.path.join(image_path, image_name)
        # print(full_image_path) # object_detection/1st_prj/train/images/123.jpg

        cv2_image = cv2.imread(full_image_path)
        H, W, C = cv2_image.shape
        # print(H, W, C)
        txt_path = os.path.join(label_path, base_name + ".txt")

        all_class_ids = []
        all_boxes = []

        with open(txt_path, "r") as f:
            lines = f.readlines()
            print(f"{image_name}: {len(lines)}")

            for line in lines:
                parts = line.split()
                class_id = int(parts[0])

                pixel_cx = float(parts[1]) * W
                pixel_cy = float(parts[2]) * H
                pixel_w = float(parts[3]) * W
                pixel_h = float(parts[4]) * H

                x_min = int(pixel_cx - (pixel_w / 2))
                y_min = int(pixel_cy - (pixel_h / 2))
                x_max = int(pixel_cx + (pixel_w / 2))
                y_max = int(pixel_cy + (pixel_h / 2))

                all_class_ids.append(class_id)
                all_boxes.append([x_min, y_min, x_max, y_max])

        # for i, box in enumerate(all_boxes):
        #     x_min, y_min, x_max, y_max = box
        #     class_id = all_class_ids[i]

        #     cv2.rectangle(cv2_image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 3)

        # dest_path = os.path.join(common_path + "/bnd_img", "bnd_" + image_name)
        # cv2.imwrite(dest_path, cv2_image)

imgi_33_images_jpg.rf.aaRknJVjki1oshmpwjhs.jpg: 2
imgi_31_images_jpg.rf.CwOn2Hvy6qIxxEhJTdiO.jpg: 6
imgi_23_images_jpg.rf.GPYJqnN00yilXVZiaMT4.jpg: 7
imgi_35_images_jpg.rf.EZjwswW6Mg5g8O1YK9wI.jpg: 2
imgi_39_images_jpg.rf.zMQXju8DeZUimxiKkTMl.jpg: 5
imgi_27_images_jpg.rf.xYAYtmS3CLFZALU1NVPQ.jpg: 13
imgi_19_images_jpg.rf.eCw82J0p6K9hKv0eoesM.jpg: 10
imgi_15_images_jpg.rf.u8VIajXCvWsMloeLB6er.jpg: 5
imgi_41_images_jpg.rf.fe3o3D7e5EktMZwwy9EX.jpg: 3
imgi_43_images_jpg.rf.HL9E8S6f85HEK7BtMojT.jpg: 8
imgi_29_images_jpg.rf.PcqD3UFS7Gdq7YDV12lY.jpg: 6
imgi_17_images_jpg.rf.Xdui7eI3Da3fENjhvGXO.jpg: 0
imgi_21_images_jpg.rf.FseLj38lQhVxaTXt2YTg.jpg: 2
imgi_37_images_jpg.rf.ZND2nV5ps1YWZaf1rSgL.jpg: 9
imgi_25_images_jpg.rf.mUlJvlX8zr1OSYA0e44c.jpg: 5


In [16]:
import os
import glob
import cv2
import torch
from torch.utils.data import Dataset
import numpy as np

class GarbageDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.img_paths = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))
        self.label_paths = sorted(glob.glob(os.path.join(labels_dir, "*.txt")))

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        current_img_path = self.img_paths[idx]
        current_label_path = self.label_paths[idx]

        cv2_image = cv2.imread(current_img_path)
        H, W, C = cv2_image.shape

        all_class_ids = []
        all_boxes = []

        with open(current_label_path, "r") as f:
            lines = f.readlines()
            for line in lines:
                parts = line.split()

                class_id = int(parts[0])

                pixel_cx = float(parts[1]) * W
                pixel_cy = float(parts[2]) * H
                pixel_w = float(parts[3]) * W
                pixel_h = float(parts[4]) * H

                x_min = int(pixel_cx - (pixel_w / 2))
                y_min = int(pixel_cy - (pixel_h / 2))
                x_max = int(pixel_cx + (pixel_w / 2))
                y_max = int(pixel_cy + (pixel_h / 2))

                all_class_ids.append(class_id)
                all_boxes.append([x_min, y_min, x_max, y_max])


        if len(all_boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(all_boxes, dtype=torch.float32)
            labels = torch.tensor(all_class_ids, dtype=torch.int64)
        
        target = {
            "boxes": boxes, 
            "labels": labels
        }

        transposed_img = cv2_image.transpose(2, 0, 1)
        image_tensor = torch.tensor(transposed_img, dtype=torch.float32) / 255.0

        return image_tensor, target

In [17]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    # zip을 이용해 이미지들만 따로 묶고, target들만 따로 묶어서 튜플(또는 리스트)로 만듭니다.
    return tuple(zip(*batch))

train_dataset = GarbageDataset(
    images_dir=image_path, 
    labels_dir=label_path
)

train_loader = DataLoader(
    train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn
)

In [18]:
images, targets = next(iter(train_loader))
print(len(images))
print(len(targets))

print(images[0].shape)
print(targets[0]["boxes"])

2
2
torch.Size([3, 276, 723])
tensor([[550., 155., 605., 198.],
        [377., 160., 494., 226.]])


In [19]:
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)

# 내 데이터셋의 클래스 개수에 맞게 헤드(출력층) 바꾸기
# 우리 데이터셋의 클래스가 만약 3개(can, pet, trash)라면, 배경을 포함해서 총 4개여야 합니다.
num_classes = 4

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Faster R-CNN은 학습 모드(train)일 때와 추론 모드(eval)일 때 뱉어내는 출력물이 완전히 다르다.
model.train()

images_list = list(img for img in images)
targets_list = list(tar for tar in targets)

loss_dict = model(images_list, targets_list)

print(f"Loss 종류들:\n{loss_dict}")

Loss 종류들:
{'loss_classifier': tensor(1.3860, grad_fn=<NllLossBackward0>), 'loss_box_reg': tensor(0.0824, grad_fn=<DivBackward0>), 'loss_objectness': tensor(0.0606, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_rpn_box_reg': tensor(0.0062, grad_fn=<DivBackward0>)}


In [21]:
import torch.optim as optim
import time

# device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
device = torch.device('cpu')
print(f"device: {device}")

model.to(device)

# Faster R-CNN 계열은 SGD나 AdamW를 많이 씁니다.
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 5
# print(time.time())
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    start_time = time.time()

    for images, targets in train_loader:

        # print(time.time())
        images_list = [img.to(device) for img in images]
        targets_list = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        loss_dict = model(images_list, targets_list)

        losses = sum(loss for loss in loss_dict.values())

        losses.backward()

        optimizer.step()

        epoch_loss += losses.item() * len(images_list)
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"소요 시간: {elapsed_time:.4f}초")
    print(f"Epoch {epoch+1}/{num_epochs} - Avg Loss: {epoch_loss / len(train_dataset):.4f}")

device: cpu
소요 시간: 44.5340초
Epoch 0/5 - Avg Loss: 0.8464
소요 시간: 43.0119초
Epoch 1/5 - Avg Loss: 0.5968
소요 시간: 44.3458초
Epoch 2/5 - Avg Loss: 0.5136
소요 시간: 44.6781초
Epoch 3/5 - Avg Loss: 0.3968
소요 시간: 44.5572초
Epoch 4/5 - Avg Loss: 0.3064
